# CodingSSM Phase 1: Code Pretraining on Kaggle T4

This is **Phase 1** of the training roadmap: pretraining CodingSSM on raw Python code
from [The Stack](https://huggingface.co/datasets/bigcode/the-stack) before any fine-tuning.

Pretraining on code teaches the model Python syntax, idioms, and structure — the
foundation that SFT and GRPO build on top of.

**Expected runtime**: ~9 hours per Kaggle session (resumes automatically from last checkpoint).

**Target**: 2 billion tokens total across multiple sessions. Progress is saved to
`pgalyen1987/RS-Code-SSM-1.6B` on HuggingFace after each checkpoint so the next
session can pick up exactly where this one left off.

**Data**: Streams Python files directly from HuggingFace — no local data download needed.
Falls back to `codeparrot/github-code` if The Stack is unavailable.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import subprocess, sys, torch

# Suppress Kaggle debugger file-validation noise
import os
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

def pip(*args, timeout=120):
    try:
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', *args],
            capture_output=True, text=True, timeout=timeout,
        )
    except subprocess.TimeoutExpired:
        print(f'  TIMEOUT {args[0]} (>{timeout}s)')
        return False
    if r.returncode != 0:
        print(f'  WARN {args[0]}: {r.stderr[-300:].strip()}')
        return False
    print(f'  OK {args[0]}')
    return True

# Standard packages
pip('safetensors', 'transformers', 'datasets', 'huggingface_hub', 'faiss-cpu', 'sentence-transformers')

# mamba-ssm — installs causal-conv1d as a dependency, so no need to install separately
# Compiles CUDA extensions from source (~4 min), 10 min hard timeout
print(f'Building mamba-ssm for torch{torch.__version__} / cuda{torch.version.cuda}...')
pip('mamba-ssm>=2.2.2', timeout=600)

print('Dependencies installed')

In [ ]:
# ── Cell 2: Auth + clone repo from HuggingFace ────────────────────────────────
import os, subprocess, sys
from pathlib import Path

REPO = 'pgalyen1987/RS-Code-SSM'

# Try all common Kaggle secret names for HF token
HF_TOKEN = ''
try:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    for label in ('HF_TOKEN', 'hf_token', 'HUGGINGFACE_TOKEN', 'huggingface_token'):
        try:
            HF_TOKEN = client.get_secret(label)
            if HF_TOKEN:
                print(f'HuggingFace: authenticated (secret: {label!r})')
                break
        except Exception:
            pass
    if not HF_TOKEN:
        print('WARNING: No HF secret found. Tried: HF_TOKEN, hf_token, HUGGINGFACE_TOKEN')
        print('  → In Kaggle: Add-ons → Secrets → enable your HF token for this notebook')
except Exception as e:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    if HF_TOKEN:
        print('HuggingFace: authenticated (env)')
    else:
        print(f'WARNING: HF_TOKEN not set ({e})')

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)

# Clone repo (only if not already inside it)
if not Path('arch').exists():
    if not Path('RS-Code-SSM').exists():
        subprocess.run(['git', 'clone', f'https://github.com/{REPO}.git', 'RS-Code-SSM'], check=True)
    os.chdir('RS-Code-SSM')
    sys.path.insert(0, '.')
    print(f'Repo: {Path.cwd()}')

# NOTE: No training data download — pretraining streams directly from HuggingFace datasets
print('Ready. Pretraining will stream Python code directly from HuggingFace.')

In [ ]:
# ── Cell 3: Check GPU ─────────────────────────────────────────────────────────
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {DEVICE}')

In [ ]:
# ── Cell 4: Download checkpoint if exists ─────────────────────────────────────
from pathlib import Path

CKPT_DIR = Path('checkpoints/pretrain')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

resume_path = None

if HF_TOKEN:
    try:
        from huggingface_hub import hf_hub_download
        local = hf_hub_download(
            repo_id='pgalyen1987/RS-Code-SSM-1.6B',
            filename='pretrain_checkpoint.pt',
            repo_type='model',
            token=HF_TOKEN,
            local_dir=str(CKPT_DIR),
        )
        resume_path = Path(local)
        # Peek at metadata
        import torch as _torch
        meta = _torch.load(resume_path, map_location='cpu')
        tokens_seen = meta.get('tokens_seen', 0)
        step = meta.get('step', 0)
        print(f'Checkpoint found: {resume_path}')
        print(f'  step={step:,}  tokens_seen={tokens_seen/1e6:.1f}M')
        del meta
    except Exception as e:
        if 'not found' in str(e).lower() or '404' in str(e) or 'Entry Not Found' in str(e):
            print('Starting fresh pretraining (no checkpoint found in HF repo)')
        else:
            print(f'WARNING: checkpoint download failed ({e})')
            print('Starting fresh pretraining')
else:
    print('No HF_TOKEN — skipping checkpoint download')
    print('Starting fresh pretraining')

In [ ]:
# ── Cell 5: Run pretraining ────────────────────────────────────────────────────
import subprocess, sys, os
from pathlib import Path

cmd = [
    sys.executable, '-m', 'train.pretrain',
    '--output-dir', 'checkpoints/pretrain',
    '--model-size', '700m',
    '--max-tokens', '2000000000',
    '--batch-size', '1',
    '--grad-accum', '16',
    '--seq-len', '1024',
    '--lr', '1e-3',
    '--device', DEVICE,
]

# Only pass --resume if we actually have a checkpoint path
if resume_path:
    cmd += ['--resume', str(resume_path)]

subprocess.run(
    cmd,
    check=True,
    env={
        **os.environ,
        'HF_TOKEN': HF_TOKEN,
        'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
        'PYDEVD_DISABLE_FILE_VALIDATION': '1',
    },
)

In [ ]:
# ── Cell 6: Upload final checkpoint to HuggingFace ────────────────────────────
import os
from pathlib import Path

CKPT_DIR = Path('checkpoints/pretrain')
HF_REPO = 'pgalyen1987/RS-Code-SSM-1.6B'

if not HF_TOKEN:
    print('No HF_TOKEN set — skipping upload')
else:
    # Find the most recent checkpoint
    candidates = sorted(CKPT_DIR.glob('*.pt'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        print('No checkpoints found in', CKPT_DIR)
    else:
        latest = candidates[0]
        print(f'Uploading {latest.name} → {HF_REPO}/pretrain_checkpoint.pt ...')
        try:
            from huggingface_hub import HfApi
            import torch as _torch

            api = HfApi(token=HF_TOKEN)
            api.upload_file(
                path_or_fileobj=str(latest),
                path_in_repo='pretrain_checkpoint.pt',
                repo_id=HF_REPO,
                repo_type='model',
            )

            # Report tokens_seen from the checkpoint
            meta = _torch.load(latest, map_location='cpu')
            tokens_seen = meta.get('tokens_seen', 0)
            step = meta.get('step', 0)
            del meta

            print(f'Upload complete!')
            print(f'  step={step:,}  tokens_seen={tokens_seen/1e6:.1f}M / 2000M')
            print(f'  https://huggingface.co/{HF_REPO}')
        except Exception as e:
            print(f'ERROR during upload: {e}')